# TradeSense · TGNN · Stage B1 — availability probe (headers only)

Checks only: **2024-08-01, 2024-09-01, 2024-10-01** × **BTCUSDT, ETHUSDT** × **incremental_book_L2, book_snapshot_25, trades** (18 files).

- HTTP HEAD first; only if HEAD does not succeed, a streamed GET whose body is never read. Every response is closed.
- **Nothing is downloaded or saved.** No Drive access, no graph tensors, no training, no model artifacts touched.
- A successful response does **not** prove a file is complete, free to use, or covered by the licence.
- Just run the one code cell below and paste its whole output to Claude.

In [ ]:
# TradeSense · TGNN · Stage B1 (v2): availability probe for extra Tardis days.
# STRICTLY HEADERS-ONLY: HTTP HEAD first; only if HEAD does not return success, a streamed GET whose body is NEVER read.
# Nothing is downloaded or saved. Every response is closed. A success status does NOT prove the file is complete, free to use or covered by the licence.
import os, requests, pandas as pd
from IPython.display import display
BASE_URL = os.environ.get('TARDIS_BASE_URL', 'https://datasets.tardis.dev')     # env var only used by my mock-server tests
DAYS = ['2024-08-01', '2024-09-01', '2024-10-01']; SYMBOLS = ['BTCUSDT', 'ETHUSDT']; TYPES = ['incremental_book_L2', 'book_snapshot_25', 'trades']
HEADERS = {'User-Agent': 'TradeSense-research-notebook/1.0 (final-year project)'}
OK = (200, 206)
def size_mb(h):
    try: return round(int(h['Content-Length']) / 1e6, 3) if 'Content-Length' in h else None
    except ValueError: return None
def probe(url):
    out = {'HEAD': None, 'GET_fallback': None, 'method_used': None, 'MB': None, 'error': ''}
    try:
        with requests.head(url, headers=HEADERS, allow_redirects=True, timeout=60) as r:    # HEAD never has a body
            out['HEAD'] = r.status_code
            if r.status_code in OK: out['method_used'] = 'HEAD'; out['MB'] = size_mb(r.headers); return out
    except Exception as e: out['error'] = 'HEAD ' + type(e).__name__ + ': ' + str(e)[:100]
    try:
        with requests.get(url, headers=HEADERS, stream=True, allow_redirects=True, timeout=60) as r:    # stream=True: body is not fetched; never read here; closed on leaving the with-block
            out['GET_fallback'] = r.status_code
            if r.status_code in OK: out['method_used'] = 'GET (body not read)'; out['MB'] = size_mb(r.headers)
    except Exception as e: out['error'] += (' | ' if out['error'] else '') + 'GET ' + type(e).__name__ + ': ' + str(e)[:100]
    return out
rows = []
for day in DAYS:
    y, m, d = day.split('-')
    for sym in SYMBOLS:
        for dt in TYPES:
            p = probe(f'{BASE_URL}/v1/binance/{dt}/{y}/{m}/{d}/{sym}.csv.gz')
            rows.append({'day': day, 'symbol': sym, 'type': dt, **p, 'reachable_by_headers': p['method_used'] is not None and (p['MB'] is None or p['MB'] >= 0.001)})
tab = pd.DataFrame(rows); display(tab)
sets = tab.groupby(['day', 'symbol'])['reachable_by_headers'].all().rename('all 3 files reachable by headers')
print('\nPER-SET RESULT (header-level observation only):'); print(sets.to_string())
print(f"\nSets with all three files reachable by headers: {int(sets.sum())} of {len(sets)} | sum of sizes reported in headers {tab['MB'].fillna(0).sum()/1000:.2f} GB (compressed; a missing size counts as 0)")
print('\nQUALIFICATIONS: a success status is NOT proof that a file is complete, free to use, or covered by the licence; completeness needs the audit after a separate approved download, and the licence needs the Tardis terms.')
print('A HEAD failure alone is not proof of absence (HEAD returned 404 earlier for files that a GET then found); only a failure on both HEAD and GET is reported as unreachable.')
print('NOTHING WAS DOWNLOADED OR SAVED; no response body was read. Paste this output to Claude; do NOT try other URLs. Downloading needs your separate approval.')


,day,symbol,type,HEAD,GET_fallback,method_used,MB,error,reachable_by_headers
0,2024-08-01,BTCUSDT,incremental_book_L2,404,200,GET (body not read),228.265,,True
1,2024-08-01,BTCUSDT,book_snapshot_25,404,200,GET (body not read),56.244,,True
2,2024-08-01,BTCUSDT,trades,404,200,GET (body not read),24.298,,True
3,2024-08-01,ETHUSDT,incremental_book_L2,404,200,GET (body not read),206.144,,True
4,2024-08-01,ETHUSDT,book_snapshot_25,404,200,GET (body not read),63.255,,True
5,2024-08-01,ETHUSDT,trades,404,200,GET (body not read),18.372,,True
6,2024-09-01,BTCUSDT,incremental_book_L2,404,200,GET (body not read),146.608,,True
7,2024-09-01,BTCUSDT,book_snapshot_25,404,200,GET (body not read),42.230,,True
8,2024-09-01,BTCUSDT,trades,404,200,GET (body not read),18.454,,True
9,2024-09-01,ETHUSDT,incremental_book_L2,404,200,GET (body not read),133.081,,True



PER-SET RESULT (header-level observation only):
day         symbol 
2024-08-01  BTCUSDT    True
            ETHUSDT    True
2024-09-01  BTCUSDT    True
            ETHUSDT    True
2024-10-01  BTCUSDT    True
            ETHUSDT    True

Sets with all three files reachable by headers: 6 of 6 | sum of sizes reported in headers 1.54 GB (compressed; a missing size counts as 0)

QUALIFICATIONS: a success status is NOT proof that a file is complete, free to use, or covered by the licence; completeness needs the audit after a separate approved download, and the licence needs the Tardis terms.
A HEAD failure alone is not proof of absence (HEAD returned 404 earlier for files that a GET then found); only a failure on both HEAD and GET is reported as unreachable.
NOTHING WAS DOWNLOADED OR SAVED; no response body was read. Paste this output to Claude; do NOT try other URLs. Downloading needs your separate approval.
